# Projet 1 : fine-tuning YOLOv8 sur Google Colab

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omotolaaa7/-traffic-vehicle-detection-yolov8/blob/main/notebooks/colab_entrainement.ipynb)

Ce notebook exécute toute la chaîne du projet sur une machine Colab (GPU gratuit),
sans rien installer sur votre ordinateur :

1. clonage du dépôt GitHub ;
2. téléchargement du sous-ensemble BMD-45 (~700 Mo, rapide depuis Colab) ;
3. import et remappage des classes ;
4. fine-tuning de YOLOv8 sur GPU ;
5. comparaison pré-entraîné / fine-tuné ;
6. sauvegarde des poids et des résultats sur votre Google Drive.

**Avant de commencer** : menu *Exécution → Modifier le type d'exécution → GPU T4*.

Attention : une session Colab est **éphémère** : tout fichier non copié sur Drive est perdu
à la fermeture. La dernière cellule s'occupe de la sauvegarde ; exécutez-la avant
de partir.

In [ ]:
# Vérifier que le GPU est bien actif (sinon : Exécution > Modifier le type d'exécution)
!nvidia-smi

In [ ]:
# 1. Cloner le dépôt du projet
!git clone https://github.com/Omotolaaa7/-traffic-vehicle-detection-yolov8.git projet1
%cd projet1

In [ ]:
# 2. Installer les dépendances manquantes.
# Colab fournit déjà torch + CUDA : ne PAS réinstaller torch depuis requirements.txt,
# on installe seulement ce qui manque.
!pip install -q ultralytics datasets pyyaml polars

## Google Drive (recommandé)

Monter Drive permet : (a) de **réutiliser** un sous-ensemble déjà téléchargé lors
d'une session précédente au lieu de tout retélécharger, (b) de **conserver** les
poids entraînés et les résultats.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER_DRIVE = '/content/drive/MyDrive/Projet1_AMA'
os.makedirs(DOSSIER_DRIVE, exist_ok=True)
print('Sauvegardes dans :', DOSSIER_DRIVE)

In [ ]:
# 3. Récupérer le dataset : depuis Drive s'il y est déjà, sinon téléchargement.
import os

archive_drive = f'{DOSSIER_DRIVE}/bmd45_subset.zip'

if os.path.exists(archive_drive):
    print('Archive trouvée sur Drive, restauration...')
    !unzip -q {archive_drive} -d data/raw/
else:
    print('Téléchargement du sous-ensemble depuis Hugging Face (~10 min)...')
    !python scripts/download_bmd45_subset.py --train 2400 --val 600 --seed 42
    print('Archivage sur Drive pour les prochaines sessions...')
    !cd data/raw && zip -q -r {archive_drive} bmd45_subset

!ls data/raw/bmd45_subset

In [ ]:
# 4. Import : remappage des 14 classes BMD-45 vers nos 4 classes alignées COCO,
# et création du split de test (dérivé de val, graine 42).
!python scripts/import_dataset.py

In [ ]:
# 5. Fine-tuning sur GPU.
# Les hyperparamètres viennent de configs/entrainement.yaml ; --device 0 force le GPU.
# Ordre de grandeur sur T4 : ~1 h pour 50 epochs sur 2400 images en 640 px.
!python scripts/train_yolo.py --device 0

In [ ]:
# 6. Comparaison pré-entraîné / fine-tuné sur le même jeu de test.
# Produit results/comparaison.md, .csv et .json : le tableau du rapport.
!python scripts/compare_models.py --device 0

In [ ]:
# Afficher le tableau comparatif
from IPython.display import Markdown, display
display(Markdown(open('results/comparaison.md', encoding='utf-8').read()))

In [ ]:
# 7. Sauvegarde, à exécuter avant de fermer la session.
# Copie sur Drive : poids fine-tunés, résultats, courbes d'entraînement.
!cp -r models/finetuned {DOSSIER_DRIVE}/ 2>/dev/null || echo 'pas de poids fine-tunés'
!cp -r results {DOSSIER_DRIVE}/
!ls -R {DOSSIER_DRIVE}

## Et après ?

- Récupérez `yolov8n_benin.pt` depuis votre Drive (`Projet1_AMA/finetuned/`) et
  placez-le dans `models/finetuned/` de votre copie locale du dépôt : l'application
  Streamlit et `detect_image.py` l'utiliseront.
- `results/comparaison.csv` contient les chiffres à reporter dans l'article
  (voir `docs/GUIDE_ARTICLE_LATEX.md`).
- Pour des détections qualitatives sur vos propres photos :
  `!python scripts/detect_image.py chemin/vers/photo.jpg --model models/finetuned/yolov8n_benin.pt`